In [ ]:
# Cell 1: Setup & Data Quality Checks
# Pre-Stage 3 checks for the Cross-Sell Intelligence Dashboard
from pyspark.sql import functions as F

print("=== Check 1: GlobalPartyId completeness in Dim_Client ===")
df_check_1 = spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN GlobalPartyId IS NULL THEN 1 ELSE 0 END) AS null_global,
        ROUND(100.0 * SUM(CASE WHEN GlobalPartyId IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_null
    FROM Gold_SalesOps_Dim_Client
""")
df_check_1.show()


In [ ]:
# Cell 2: Orphan ClientPartyId in Fact_Transaction
print("\n=== Check 2: Orphan ClientPartyId in Fact_Transaction ===")
df_check_2 = spark.sql("""
    SELECT COUNT(*) AS orphans
    FROM Gold_SalesOps_Fact_Transaction f
    LEFT JOIN Gold_SalesOps_Dim_Client c ON c.PartyId = f.ClientPartyId
    WHERE c.PartyId IS NULL
""")
df_check_2.show()


In [ ]:
# Cell 3: Penetration baseline (LOB bucket distribution)
print("\n=== Check 3: Penetration baseline (LOB bucket distribution) ===")
# Using COALESCE(GlobalPartyId, PartyId) as the unique cross-sell entity due to 77% nulls in GlobalPartyId
df_check_3 = spark.sql("""
    WITH client_lobs AS (
        SELECT
            COALESCE(dc.GlobalPartyId, dc.PartyId) AS CrossSellEntityId,
            COUNT(DISTINCT f.GlobalProductLineId) AS lob_count
        FROM Gold_SalesOps_Fact_Transaction f
        JOIN Gold_SalesOps_Dim_Client dc ON dc.PartyId = f.ClientPartyId
        WHERE f.GlobalProductLineId IS NOT NULL
        GROUP BY COALESCE(dc.GlobalPartyId, dc.PartyId)
    )
    SELECT
        CASE
            WHEN lob_count = 1 THEN '1'
            WHEN lob_count = 2 THEN '2'
            WHEN lob_count = 3 THEN '3'
            ELSE '4+'
        END AS lob_bucket,
        COUNT(*) AS client_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM client_lobs
    GROUP BY 1
    ORDER BY 1
""")
df_check_3.show()


In [ ]:
# Cell 4: Create Dim_ProductLine view
# Using PySpark CREATE OR REPLACE VIEW to persist it to the Lakehouse
print("\n=== Creating Dim_ProductLine View ===")
spark.sql("""
    CREATE OR REPLACE VIEW Gold_SalesOps_Dim_ProductLine AS
    SELECT DISTINCT
        GlobalProductLineId,
        GlobalProductLine,
        GlobalProductClassId,
        GlobalProductClass
    FROM Gold_SalesOps_Fact_Transaction
    WHERE GlobalProductLineId IS NOT NULL
""")
print("View Gold_SalesOps_Dim_ProductLine created successfully.")
